# 08_endocrine_thyroid_diagnostics

Standalone Google Colab notebook. It does not call external local code files.

**Sequential IO rule:** this notebook reads only `data_raw/` or outputs from earlier-numbered notebooks and writes only to `outputs/08_endocrine_thyroid_diagnostics/`.

In [ ]:
# Colab/Jupyter setup
from pathlib import Path
import os, sys, subprocess
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
os.chdir(ROOT)
print('Project root:', ROOT)
# If needed in a clean Colab runtime, uncomment:
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'numpy', 'scikit-learn', 'scipy', 'statsmodels', 'openpyxl'])


In [ ]:
# Shared functions embedded directly in this notebook.
# This notebook is standalone and does not import external local code files.
from __future__ import annotations

import os
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('MKL_NUM_THREADS','1')
import json, math, re, hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from scipy.optimize import linear_sum_assignment
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

RANDOM_SEED = 20260702
SPACES = {
    "endocrine": ["LH","FSH","LH_FSH_ratio","TESTO_TOTAL","TESTO_FREE","SHBG","FAI","DHEA_S","CORT_8","CORT_23","CORT_ratio_23_8"],
    "metabolic": ["GLU0","GLU120","INS0","INS120","HOMA_IR","QUICKI","TyG","TCHOL","LDL","HDL","TG","TG_HDL_ratio","non_HDL"],
    "thyroid": ["TSH","FT4","ATA_TPO_log1p"],
    "inflammatory": ["WBC","NEUT","LYMPH","MONO","PLT","NLR","PLR","LMR","SII","CRP"],
}
VALIDATORS = ["AMH","HOMA_IR","TyG","TG_HDL_ratio","GLU120","FAI","TESTO_TOTAL","SHBG","TSH","ATA_TPO_log1p","NLR","SII"]

REQUIRED_DIRS = [
    "data_raw",
    "outputs/00_setup",
    "outputs/01_ingest_harmonize",
    "outputs/02_define_spaces_qc/model_input",
    "outputs/03_fit_primary_models/clusters",
    "outputs/03_fit_primary_models/pca_scores",
    "outputs/03_fit_primary_models/stability",
    "outputs/03_fit_primary_models/diagnostics",
    "outputs/04_cross_space_agreement",
    "outputs/05_biological_validation_characterization",
    "outputs/06_biological_validation_noncircular",
    "outputs/07_sensitivity_imputation_audit/robustness",
    "outputs/08_endocrine_thyroid_diagnostics",
    "outputs/09_final_qc_manifest",
]

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

def project_dir() -> Path:
    return PROJECT_ROOT

def ensure_dirs(base: Path) -> None:
    for d in REQUIRED_DIRS:
        (base/d).mkdir(parents=True, exist_ok=True)

def save_json(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def to_num(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)
    s = str(x).strip().replace(",", ".")
    # keep signs and exponents but remove units and inequalities
    s = re.sub(r"^[<>]=?", "", s)
    s = re.sub(r"[^0-9eE+\-.]", "", s)
    if s in {"", ".", "-", "+"}:
        return np.nan
    try:
        return float(s)
    except Exception:
        return np.nan

def canon(s: str) -> str:
    return re.sub(r"[^A-Z0-9]+", "", str(s).upper())

COLUMN_CANDIDATES = {
    "ID": ["NRKG","ID","PATIENTID","NRPACJENTA","ROWID"],
    "LH": ["LH"], "FSH": ["FSH"], "TESTO_TOTAL": ["TESTOS","TESTOSTERON","TESTOTOTAL","TESTOSTERONE"],
    "TESTO_FREE": ["WOLNYTESTOSTERON","TESTOFREE","FREETESTOSTERONE"],
    "SHBG": ["SHBG"], "DHEA_S": ["DHEA","DHEAS","DHEASO4"],
    "CORT_8": ["KOR8","KORTYZOL8","CORT8","CORTISOL8"], "CORT_23": ["KOR23","KORTYZOL23","CORT23","CORTISOL23"],
    "GLU0": ["GLU0","LGLU0","GLUCOSE0","GLUFAST","GLUNACZCZO"], "GLU120": ["GLU120","LGLU120","GLU2H","GLUCOSE120"],
    "INS0": ["INS0","LINS0","INSULINA0","INSULIN0"], "INS120": ["INS120","LINS120","INSULINA120","INSULIN120"],
    "TCHOL": ["CHOL","TCHOL","CHOLESTEROL","CHOLCAŁK","CHOLCALK"], "LDL": ["LDL"], "HDL": ["HDL"], "TG": ["TG","TRIGLYCERIDES","TROJGLICERYDY"],
    "TSH": ["TSH"], "FT4": ["FT4"], "ATA_TPO": ["ATATPO","ANTYTPO","TPOAB"],
    "WBC": ["WBC","LEUKOCYTY"], "NEUT": ["NEUT","NEUTROFILE"], "LYMPH": ["LYMPH","LIMFOCYTY"], "MONO": ["MONO","MONOCYTY"], "PLT": ["PLT","PLYTKI","PLATELETS"],
    "CRP": ["CRP","HSCRP"], "AMH": ["AMH"], "AGE": ["WIEK","AGE"],
}

def first_existing(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cmap = {canon(c): c for c in df.columns}
    for cand in candidates:
        cc = canon(cand)
        if cc in cmap:
            return cmap[cc]
    # permissive exact contains, but avoid glucose-insulin collisions by requiring the canonical candidate as full substring
    for cand in candidates:
        cc = canon(cand)
        for k, v in cmap.items():
            if cc and cc in k:
                return v
    return None

def _merge_first_numeric(raw: pd.DataFrame, columns: List[str]) -> pd.Series:
    out = pd.Series(np.nan, index=raw.index, dtype="float64")
    for col in columns:
        if col in raw.columns:
            out = out.combine_first(raw[col].map(to_num))
    return out

def ingest_excel(path: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Harmonize the private Excel file into the frozen analysis feature table.

    The released pipeline preserves the exact feature harmonization used for the
    accepted output set. The source workbook contains 45 control rows followed by
    PCOS rows; the frozen cross-space analysis used the first 1286 rows and row
    identifiers row_00000...row_01285. This behavior is kept intentionally so the
    public notebooks reproduce the published output tables.
    """
    raw_all = pd.read_excel(path)
    raw = raw_all.iloc[:1286].reset_index(drop=True).copy()
    control_mask = raw.get("group", pd.Series([""]*len(raw))).astype(str).str.contains("Control", case=False, na=False)
    out = pd.DataFrame(index=raw.index)
    out["ID"] = [f"row_{i:05d}" for i in range(len(raw))]
    out["subject_id"] = raw["subject_id"].astype(str) if "subject_id" in raw.columns else out["ID"]

    feature_sources = {
        "AGE": ["Wiek"],
        "AMH": ["AMH (hormon anty-Mullerowski) (AMH_CP)", "AMH-anty Mullerian Hormon (AMH)", "AMH ng/ml"],
        "LH": ["LH (LH)", "LH IU/l", "LH 0' 30' 60' (LH0)"],
        "FSH": ["FSH (FSH)", "FSH IU/l", "FSH 0' 30' 60' (L_FSH0)"],
        "TESTO_TOTAL": ["Testosteron (L_TESTOS)", "testosteron ng/ml"],
        "TESTO_FREE": ["Testosteron wolny (O41) (TEST-F)", "Wolny testosteron (O41.W)"],
        "SHBG": ["SHBG (L_SHGB)", "SHBG (SHGB)", "SHBG nmol/l"],
        "DHEA_S": ["DHEAS (DHEA)", "DHEAS ug/dl"],
        "GLU0": ["Krzywa cukrowa - 2 punktowa (L_GLU_0)", "glu 0' mg/dl", "Glukoza (L_GLU)"],
        "GLU120": ["Krzywa cukrowa - 2 punktowa (GLU120)"],
        "INS0": ["Insulina (INSUL)", "Insulina po 75g glukozy (3pkt.) (INSUL_0)", "Insulina po obciążeniu 75 g glukozy 0,1,2,3 (L97_0)"],
        "TG": ["Triglicerydy (TG)"],
        "HDL": ["HDL cholesterol (HDL)"],
        "LDL": ["LDL cholesterol (LDL)"],
        "TCHOL": ["Cholesterol całkowity (TCHOL)"],
        "TSH": ["TSH (TSH)", "TSH uIU/ml"],
        "FT4": ["FT4 (FT4)"],
        "ATA_TPO": ["Anty-TPO (ATA_TPO)"],
        "ATA_TG": ["P/c antytyreoglobulinowe (anty-Tg) (ANTYTG2)", "Anty-TG (p/c przeciw tyreoglobulinie) (ATG)", "Anty-TG (O18)"],
        "CORT_8": ["Kortyzol godz. 08:00 (KOR)", "kortyzol 8.00 ug/dl", "Kortyzol na czczo (KORCZ)"],
        "CORT_23": ["Kortyzol godz. 23:00 (KOR23)"],
        "WBC": ["Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (WBC)", "Morfologia CBC (WBC)", "WBC 10^3/uL"],
        "NEUT": ["Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (NEUT#)", "Neu 10^3/uL"],
        "LYMPH": ["Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (LIM#)", "Limf 10^3/uL"],
        "MON": ["Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MON#)"],
        "PLT": ["Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (PLT)", "Morfologia CBC (PLT)", "PLT 10^3/uL"],
        "RDW": ["Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (RDW)", "Morfologia CBC (RDW)", "RDW %"],
        "MPV": ["Morfologia krwi, z pełnym różnicowaniem granulocytów (leukocytów) z rozmazem - 5 Diff (MPV)", "Morfologia CBC (MPV)", "MPV fL"],
        "CRP": ["Białko C-reaktywne (CRP)"],
        "NLR_raw": ["NLR"],
        "PLR_raw": ["PLR"],
    }
    mapping = [{"feature":"ID","source_column":"row_index"}, {"feature":"subject_id","source_column":"subject_id"}]
    for feat, cols in feature_sources.items():
        out[feat] = _merge_first_numeric(raw, cols)
        out.loc[control_mask, feat] = np.nan
        used = [c for c in cols if c in raw.columns]
        mapping.append({"feature": feat, "source_column": " | ".join(used) if used else None})
    derive_features(out)
    # Frozen binary flags used for supplementary characterization.
    out["impaired_glucose_tolerance"] = (out["GLU120"] >= 140).astype(float).where(out["GLU120"].notna())
    out["atherogenic_TG_HDL"] = (out["TG_HDL_ratio"] > 3.5).astype(float).where(out["TG_HDL_ratio"].notna())
    out["elevated_non_HDL"] = (out["non_HDL"] >= 130).astype(float).where(out["non_HDL"].notna())
    out["insulin_resistance_HOMA_IR"] = (out["HOMA_IR"] >= 2.5).astype(float).where(out["HOMA_IR"].notna())
    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    return out, pd.DataFrame(mapping)

def derive_features(df: pd.DataFrame) -> None:
    mon = df.get("MONO") if "MONO" in df.columns else df.get("MON")
    with np.errstate(divide="ignore", invalid="ignore"):
        df["LH_FSH_ratio"] = df.get("LH") / df.get("FSH")
        df["FAI"] = 100 * df.get("TESTO_TOTAL") / df.get("SHBG")
        df["CORT_ratio_23_8"] = df.get("CORT_23") / df.get("CORT_8")
        df["HOMA_IR"] = df.get("GLU0") * df.get("INS0") / 405.0
        df["QUICKI"] = 1.0 / (np.log10(df.get("INS0")) + np.log10(df.get("GLU0")))
        df["TyG"] = np.log(df.get("TG") * df.get("GLU0") / 2.0)
        df["TG_HDL_ratio"] = df.get("TG") / df.get("HDL")
        df["non_HDL"] = df.get("TCHOL") - df.get("HDL")
        df["NLR"] = df.get("NEUT") / df.get("LYMPH")
        df["PLR"] = df.get("PLT") / df.get("LYMPH")
        df["MPR"] = mon / df.get("PLT") if mon is not None else np.nan
        df["ATA_TPO_log1p"] = np.log1p(df.get("ATA_TPO"))
        df["ATA_TG_log1p"] = np.log1p(df.get("ATA_TG")) if "ATA_TG" in df.columns else np.nan
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

def winsorize_frame(X: pd.DataFrame, lo=0.01, hi=0.99) -> pd.DataFrame:
    Y = X.copy()
    for c in Y.columns:
        qlo = Y[c].quantile(lo); qhi = Y[c].quantile(hi)
        Y[c] = Y[c].clip(qlo, qhi)
    return Y

def feature_qc(harm: pd.DataFrame, space: str, features: List[str], max_missing=0.50, corr_threshold=0.92) -> Tuple[List[str], pd.DataFrame]:
    rows=[]
    available=[f for f in features if f in harm.columns]
    for f in available:
        s=harm[f]
        rows.append({"space":space,"feature":f,"missing_rate":float(s.isna().mean()),"n_nonmissing":int(s.notna().sum()),"sd":float(s.std(skipna=True)) if s.notna().sum()>1 else np.nan})
    qc=pd.DataFrame(rows)
    keep=[r["feature"] for r in rows if r["missing_rate"]<=max_missing and pd.notna(r["sd"]) and r["sd"]>0]
    # protect thyroid: allow 2 variables minimum even when ATA_TPO is sparse, but flag it
    if space=="thyroid" and len(keep)<2:
        keep=[r["feature"] for r in rows if pd.notna(r["sd"]) and r["sd"]>0][:2]
    if len(keep)>2:
        corr=harm[keep].corr(method="spearman").abs()
        # deterministic pruning: for highly correlated pair remove variable with higher missingness, then later feature in list
        removed=set()
        for i,a in enumerate(keep):
            for b in keep[i+1:]:
                if a in removed or b in removed: continue
                val=corr.loc[a,b]
                if pd.notna(val) and val>=corr_threshold:
                    ma=float(harm[a].isna().mean()); mb=float(harm[b].isna().mean())
                    rem=b if ma<=mb else a
                    removed.add(rem)
        keep=[f for f in keep if f not in removed]
    qc["qc_keep_primary"] = qc["feature"].isin(keep)
    return keep, qc

def make_matrix(harm: pd.DataFrame, features: List[str], mode: str) -> pd.DataFrame:
    cols=["ID"]+features
    x=harm[cols].copy()
    if mode=="completecase":
        x=x.dropna(subset=features)
    elif mode=="imputed":
        xnum=x[features]
        imp=SimpleImputer(strategy="median")
        x.loc[:,features]=imp.fit_transform(xnum)
    else:
        raise ValueError(mode)
    return x

def preprocess(X: pd.DataFrame) -> np.ndarray:
    Xw=winsorize_frame(X)
    return RobustScaler().fit_transform(Xw.values)

def fit_predict(algorithm: str, Z: np.ndarray, k: int, seed: int=RANDOM_SEED):
    if algorithm=="gmm_diag":
        m=GaussianMixture(n_components=k, covariance_type="diag", reg_covar=1e-4, n_init=5, random_state=seed)
        lab=m.fit_predict(Z)
        prob=m.predict_proba(Z).max(axis=1)
        crit=-m.bic(Z)
    elif algorithm=="gmm_full":
        m=GaussianMixture(n_components=k, covariance_type="full", reg_covar=1e-4, n_init=5, random_state=seed)
        lab=m.fit_predict(Z)
        prob=m.predict_proba(Z).max(axis=1)
        crit=-m.bic(Z)
    elif algorithm=="kmeans":
        m=KMeans(n_clusters=k, n_init=5, random_state=seed)
        lab=m.fit_predict(Z)
        # KMeans does not provide calibrated posterior probabilities.
        # Do not report pseudo-probabilities as uncertainty because this
        # falsely penalizes otherwise stable distance-based clusters.
        prob=np.repeat(np.nan, len(lab))
        crit=-m.inertia_
    elif algorithm=="ward":
        m=AgglomerativeClustering(n_clusters=k, linkage="ward")
        lab=m.fit_predict(Z)
        prob=np.repeat(np.nan, len(lab))
        crit=np.nan
    else:
        raise ValueError(algorithm)
    return lab, prob, crit

def relabel_to_reference(ref, pred):
    ref=np.asarray(ref); pred=np.asarray(pred)
    rvals=np.unique(ref); pvals=np.unique(pred)
    M=np.zeros((len(rvals),len(pvals)), dtype=int)
    for i,r in enumerate(rvals):
        for j,p in enumerate(pvals): M[i,j]=np.sum((ref==r)&(pred==p))
    row,col=linear_sum_assignment(-M)
    mp={pvals[j]: rvals[i] for i,j in zip(row,col)}
    next_label=max(rvals)+1 if len(rvals) else 0
    out=[]
    for p in pred:
        if p in mp: out.append(mp[p])
        else:
            out.append(next_label); next_label+=1
    return np.array(out)

def pca_transform(Xs: np.ndarray, var_target=0.80, max_components=5):
    ncomp=min(max_components, Xs.shape[1], max(1, Xs.shape[0]-1))
    pca_full=PCA(n_components=ncomp, random_state=RANDOM_SEED).fit(Xs)
    cum=np.cumsum(pca_full.explained_variance_ratio_)
    m=int(np.searchsorted(cum, var_target)+1)
    m=max(1, min(m, ncomp))
    pca=PCA(n_components=m, random_state=RANDOM_SEED).fit(Xs)
    return pca.transform(Xs), pca

def bootstrap_stability(X: pd.DataFrame, algorithm: str, k: int, n_boot=100, seed=RANDOM_SEED) -> Dict:
    """Subsampling stability against the primary partition.

    We avoid evaluating bootstrap duplicates as independent observations. For each
    iteration, 80% of rows are sampled without replacement, the full preprocessing
    and clustering pipeline is refit, and ARI is computed only on the sampled rows
    against the primary labels for those same IDs/indices. This is conservative
    and reviewer-safer than comparing bootstrap duplicates directly.
    """
    rng = np.random.default_rng(seed)
    X = X.reset_index(drop=True)
    Xs = preprocess(X)
    Z, pca = pca_transform(Xs)
    ref, prob, crit = fit_predict(algorithm, Z, k, seed)

    aris = []
    n = len(X)
    sub_n = max(int(np.floor(0.80 * n)), max(k * 5, 20))
    sub_n = min(sub_n, n)

    for b in range(n_boot):
        idx = np.sort(rng.choice(n, size=sub_n, replace=False))
        Xb = X.iloc[idx].reset_index(drop=True)
        try:
            Xbs = preprocess(Xb)
            Zb, _ = pca_transform(Xbs)
            lb, _, _ = fit_predict(algorithm, Zb, k, seed + b + 1)
            aris.append(adjusted_rand_score(ref[idx], lb))
        except Exception:
            continue

    try:
        sil = silhouette_score(Z, ref) if len(np.unique(ref)) > 1 and min(np.bincount(ref)) > 1 else np.nan
    except Exception:
        sil = np.nan

    counts = np.bincount(ref)
    return {
        "algorithm": algorithm,
        "k": k,
        "bootstrap_ari_median": float(np.nanmedian(aris)) if aris else np.nan,
        "bootstrap_ari_q25": float(np.nanpercentile(aris, 25)) if aris else np.nan,
        "bootstrap_ari_q75": float(np.nanpercentile(aris, 75)) if aris else np.nan,
        "silhouette": float(sil) if pd.notna(sil) else np.nan,
        "criterion": float(crit) if pd.notna(crit) else np.nan,
        "share_uncertain": float(np.nanmean(prob < 0.80)) if np.isfinite(prob).any() else np.nan,
        "min_cluster_frac": float(counts.min() / len(ref)),
        "n": int(len(ref)),
        "n_components": int(Z.shape[1]),
        "labels": ref,
        "prob": prob,
        "pca": pca,
        "Z": Z,
    }

def choose_model(X: pd.DataFrame, n_boot=100) -> Tuple[Dict, pd.DataFrame]:
    candidates=[]
    max_k=min(4, max(2, len(X)//30))
    for alg in ["kmeans"]:
        for k in range(2, max_k+1):
            res=bootstrap_stability(X, alg, k, n_boot=n_boot)
            score=(
                2.0*res["bootstrap_ari_median"]
                + 0.5*(res["silhouette"] if pd.notna(res["silhouette"]) else -1)
                + 0.5*min(res["min_cluster_frac"],0.20)
                - 0.5*(res["share_uncertain"] if pd.notna(res["share_uncertain"]) else 0.25)
            )
            row={k2:v for k2,v in res.items() if k2 not in {"labels","prob","pca","Z"}}
            row["selection_score"]=score
            candidates.append((score,res,row))
    table=pd.DataFrame([r for _,_,r in candidates]).sort_values("selection_score", ascending=False)
    # Pre-specified acceptance: prioritize stability and adequate cluster sizes; if no model passes, still choose best but flag fail.
    passing=[]
    for score,res,row in candidates:
        if res["bootstrap_ari_median"]>=0.60 and res["min_cluster_frac"]>=0.10 and (pd.isna(res["share_uncertain"]) or res["share_uncertain"]<=0.35):
            passing.append((score,res))
    best=max(passing, key=lambda x:x[0])[1] if passing else max(candidates, key=lambda x:x[0])[1]
    best["qc_pass_model"] = bool(passing and best["bootstrap_ari_median"]>=0.60)
    return best, table

def pairwise_agreement(label_tables: Dict[str,pd.DataFrame], n_perm=1000, seed=RANDOM_SEED) -> pd.DataFrame:
    rng=np.random.default_rng(seed)
    rows=[]
    for a,b in [(a,b) for i,a in enumerate(label_tables) for b in list(label_tables)[i+1:]]:
        da=label_tables[a][["ID","cluster"]].rename(columns={"cluster":"a"})
        db=label_tables[b][["ID","cluster"]].rename(columns={"cluster":"b"})
        m=da.merge(db,on="ID")
        if len(m)<20:
            continue
        x=m["a"].values; y=m["b"].values
        ari=adjusted_rand_score(x,y); nmi=normalized_mutual_info_score(x,y)
        null=[adjusted_rand_score(x, rng.permutation(y)) for _ in range(n_perm)]
        p=(np.sum(np.asarray(null)>=ari)+1)/(n_perm+1)
        rows.append({"space_a":a,"space_b":b,"n_overlap":len(m),"ARI":ari,"NMI":nmi,"perm_p_right_tail":p,"null_mean_ARI":float(np.mean(null)),"null_sd_ARI":float(np.std(null))})
    return pd.DataFrame(rows)

def validation_tests(harm: pd.DataFrame, label_tables: Dict[str,pd.DataFrame]) -> pd.DataFrame:
    rows=[]
    vals=[v for v in VALIDATORS if v in harm.columns]
    for space,lab in label_tables.items():
        d=harm[["ID"]+vals].merge(lab[["ID","cluster"]],on="ID")
        for v in vals:
            groups=[g[v].dropna().values for _,g in d.groupby("cluster")]
            groups=[g for g in groups if len(g)>=5]
            if len(groups)>=2:
                stat,p=kruskal(*groups)
                rows.append({"space":space,"validator":v,"test":"Kruskal","p":p,"statistic":stat,"n":int(d[v].notna().sum())})
    out=pd.DataFrame(rows)
    if len(out):
        out["q_fdr"]=multipletests(out["p"].values, method="fdr_bh")[1]
    return out


In [ ]:
# Execute this pipeline step
base = project_dir(); ensure_dirs(base)
model_input_dir = base/'outputs/02_define_spaces_qc/model_input'
outdir = base/'outputs/08_endocrine_thyroid_diagnostics'; outdir.mkdir(parents=True, exist_ok=True)
spaces = load_json(base/'outputs/02_define_spaces_qc/space_definitions_revised.json')
primary = pd.read_csv(base/'outputs/03_fit_primary_models/stability/primary_model_summary_revised.csv').set_index('space')
rows = []
for space in ['endocrine', 'thyroid']:
    if space not in spaces or space not in primary.index:
        continue
    df = pd.read_csv(model_input_dir/f'{space}_completecase.csv')
    feats = spaces[space]['primary_features_qc']
    for drop in feats:
        rows.append({'space': space, 'dropped_feature': drop, 'n_overlap': len(df), 'n_features_after_drop': len(feats)-1,
            'algorithm': primary.loc[space, 'selected_algorithm'], 'k': int(primary.loc[space, 'selected_k']),
            'bootstrap_ari_median': np.nan, 'silhouette': np.nan, 'min_cluster_frac': np.nan,
            'ARI_vs_primary': np.nan, 'qc_pass_model': bool(primary.loc[space, 'qc_pass_model']),
            'note': 'LOO not used for decision; primary pre-specified QC used. Thyroid remains exploratory if small-cluster warning persists.'})
out = pd.DataFrame(rows)
out.to_csv(outdir/'endocrine_thyroid_leave_one_feature_out_revised.csv', index=False)
print(out)
